In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

def generate_pareto(size, theta_val):
    u = np.random.uniform(0, 1, size=size)
    return np.power(1 - u, 1 / (1 - theta_val))

POPULATION_THETA = 3.0
SAMPLE_SIZE = 100
CONFIDENCE = 0.95

In [ ]:
obs_data = generate_pareto(SAMPLE_SIZE, POPULATION_THETA)

avg_log_x = np.log(obs_data).mean()
mle_theta = 1 + (1 / avg_log_x)
mle_median = np.power(2, 1 / (mle_theta - 1))

real_median = 2**(1 / (POPULATION_THETA - 1))

z_crit = norm.ppf(1 - (1 - CONFIDENCE) / 2)

theta_error = z_crit / (np.sqrt(SAMPLE_SIZE) * avg_log_x)
theta_ci_asymp = [mle_theta - theta_error, mle_theta + theta_error]

median_error = (z_crit * np.log(2) * mle_median) / (np.sqrt(SAMPLE_SIZE) * (mle_theta - 1))
median_ci_asymp = [mle_median - median_error, mle_median + median_error]

print(f"Theta: {mle_theta:.4f} (True: {POPULATION_THETA})")
print(f"Median: {mle_median:.4f} (True: {real_median:.4f})")

In [ ]:
BOOT_REPS = 50000
p_boot_theta = np.zeros(BOOT_REPS)
p_boot_median = np.zeros(BOOT_REPS)

for i in range(BOOT_REPS):
    b_sample = generate_pareto(SAMPLE_SIZE, mle_theta)
    
    b_theta = 1 + 1 / np.log(b_sample).mean()
    p_boot_theta[i] = b_theta - mle_theta
    p_boot_median[i] = 2**(1 / (b_theta - 1)) - mle_median

t_low, t_high = np.percentile(p_boot_theta, [2.5, 97.5])
m_low, m_high = np.percentile(p_boot_median, [2.5, 97.5])

ci_theta_p_boot = [mle_theta - t_high, mle_theta - t_low]
ci_median_p_boot = [mle_median - m_high, mle_median - m_low]

In [ ]:
NP_BOOT_REPS = 1000
np_boot_theta = np.empty(NP_BOOT_REPS)
np_boot_median = np.empty(NP_BOOT_REPS)

for j in range(NP_BOOT_REPS):
    resample = np.random.choice(obs_data, size=SAMPLE_SIZE, replace=True)
    
    res_theta = 1 + 1 / np.log(resample).mean()
    np_boot_theta[j] = res_theta - mle_theta
    np_boot_median[j] = 2**(1 / (res_theta - 1)) - mle_median

nt_low, nt_high = np.percentile(np_boot_theta, [2.5, 97.5])
nm_low, nm_high = np.percentile(np_boot_median, [2.5, 97.5])

ci_theta_np_boot = [mle_theta - nt_high, mle_theta - nt_low]
ci_median_np_boot = [mle_median - nm_high, mle_median - nm_low]

In [ ]:
summary_list = [
    {
        "Цель": "Параметр Theta",
        "Метод": "Asymptotic MLE",
        "Нижняя": theta_ci_asymp[0],
        "Верхняя": theta_ci_asymp[1]
    },
    {
        "Цель": "Параметр Theta",
        "Метод": "Parametric Bootstrap",
        "Нижняя": ci_theta_p_boot[0],
        "Верхняя": ci_theta_p_boot[1]
    },
    {
        "Цель": "Параметр Theta",
        "Метод": "Non-Parametric Bootstrap",
        "Нижняя": ci_theta_np_boot[0],
        "Верхняя": ci_theta_np_boot[1]
    },
    {
        "Цель": "Медиана",
        "Метод": "Asymptotic",
        "Нижняя": median_ci_asymp[0],
        "Верхняя": median_ci_asymp[1]
    },
    {
        "Цель": "Медиана",
        "Метод": "Parametric Bootstrap",
        "Нижняя": ci_median_p_boot[0],
        "Верхняя": ci_median_p_boot[1]
    },
    {
        "Цель": "Медиана",
        "Метод": "Non-Parametric Bootstrap",
        "Нижняя": ci_median_np_boot[0],
        "Верхняя": ci_median_np_boot[1]
    }
]

final_df = pd.DataFrame(summary_list)
final_df["Ширина"] = final_df["Верхняя"] - final_df["Нижняя"]

final_df["Covered"] = False
final_df.loc[final_df["Цель"] == "Параметр Theta", "Covered"] = \
    (final_df["Нижняя"] <= POPULATION_THETA) & (final_df["Верхняя"] >= POPULATION_THETA)
final_df.loc[final_df["Цель"] == "Медиана", "Covered"] = \
    (final_df["Нижняя"] <= real_median) & (final_df["Верхняя"] >= real_median)

final_df.round(4)